# Workspace Setup (Admin-only)

This notebook provisions shared resources for the course. Run it once per workspace before students begin.

**What it creates:**
1. Shared Unity Catalog namespace (`dbacademy.airbnb_agent`) with volumes and PDFs
2. A shared Knowledge Assistant with the listings PDF attached
3. User permissions on all shared resources

In [0]:
%pip install "reportlab==4.4.10" "databricks-sdk==0.110.0" "pyyaml==6.0.3"
dbutils.library.restartPython()

In [0]:
import os
import shutil

from databricks.sdk import WorkspaceClient

from _lib.agent_bricks_manager import AgentBricksManager

ws = WorkspaceClient()
abm = AgentBricksManager(ws)

## 1. Define Variables and Create Catalog / Schema / Volume

Set up the shared UC namespace and define all variable names used throughout this notebook.

In [0]:
# --- Shared UC namespace ---
SHARED_CATALOG = "dbacademy"
SHARED_SCHEMA  = "airbnb_agent"
SHARED_VOLUME  = "ka_docs"

# --- Knowledge Assistant ---
KA_NAME = "Airbnb_Knowledge_Assistant"

# --- External (Marketplace) dependencies ---
# Loaded outside this notebook from the Marketplace share; the grant in
# Section 5 assumes the table is already present.
DATABRICKS_SHARE_NAME = "dbacademy_airbnb"
TABLE_NAME            = "sf_airbnb_listings"

# --- Volume paths ---
VOLUME_PATH = f"/Volumes/{SHARED_CATALOG}/{SHARED_SCHEMA}/{SHARED_VOLUME}"

# Create catalog / schema / volume
spark.sql(f"USE CATALOG {SHARED_CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SHARED_SCHEMA}")
spark.sql(f"USE SCHEMA {SHARED_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SHARED_SCHEMA}.{SHARED_VOLUME}")

print(f"Shared catalog : {SHARED_CATALOG}")
print(f"Shared schema  : {SHARED_SCHEMA}")
print(f"Shared volume  : {VOLUME_PATH}")

## 2. Load Airbnb data into shared catalog

In [0]:
from pyspark.sql.functions import concat, lit, col, regexp_replace, trim

df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .load(f"/Volumes/{DATABRICKS_SHARE_NAME}/v01/sf-listings/sf-airbnb.csv")
    .select("id", "name", "neighbourhood", "neighbourhood_cleansed", "summary", "price", "room_type")
    .withColumn("price", regexp_replace(trim(col("price")), "[$,]", "").cast("double"))
    .withColumn(
        "listing_source_information",
        concat(
            lit("ID of the property: "), col("id"),
            lit("\nName of the property: "), col("name"),
            lit("\nSummary of the property: "), col("summary"),
        ),
    )
    .limit(50)
)

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)
print(f"Loaded {df.count()} rows into {SHARED_CATALOG}.{SHARED_SCHEMA}.{TABLE_NAME}")

## 3. Generate listings PDF into shared volume

In [0]:
from _lib.pdf_creation import create_listings_pdf

pdf_path = f"{VOLUME_PATH}/airbnb_listings.pdf"

print("Generating Airbnb listings PDF...")
create_listings_pdf(
    table_name=TABLE_NAME,
    output_path=pdf_path,
    rows_limit=50,
)

## 4. Create the Shared Knowledge Assistant

Create the **Airbnb Knowledge Assistant** and attach the listings PDF volume as a knowledge source.

In [0]:
existing_ka = abm.find_by_name(KA_NAME)

if existing_ka:
    ka_tile_id = existing_ka.tile_id
    print(f"Knowledge Assistant already exists: {KA_NAME} (tile_id: {ka_tile_id})")
else:
    print(f"Creating Knowledge Assistant: {KA_NAME}...")
    knowledge_sources = AgentBricksManager.ka_get_knowledge_sources_from_volumes([
        (VOLUME_PATH, "Airbnb listings documentation"),
    ])

    ka_result = abm.ka_create(
        name=KA_NAME,
        knowledge_sources=knowledge_sources,
        description="Knowledge Assistant for Airbnb listings data. Ask questions about property details, pricing, and neighborhoods.",
        instructions="You are a helpful assistant that answers questions about Airbnb listings in San Francisco. Use the provided documents to find relevant information about properties, pricing, and neighborhoods.",
    )
    ka_tile_id = ka_result["knowledge_assistant"]["tile"]["tile_id"]
    print(f"  Created KA with tile_id: {ka_tile_id}")

## 5. Share the KA and grant read access to shared resources

In [0]:
# Share KA with all users via the standard Permissions API (CAN_MANAGE)
import requests as _requests

_headers = ws.config.authenticate()
_headers["Content-Type"] = "application/json"
resp = _requests.patch(
    f"{ws.config.host}/api/2.0/permissions/knowledge-assistants/{ka_tile_id}",
    headers=_headers,
    json={
        "access_control_list": [
            {"group_name": "users", "permission_level": "CAN_MANAGE"}
        ]
    },
    timeout=30,
)
resp.raise_for_status()
print(f"Shared KA '{KA_NAME}' with all users (CAN_MANAGE)")

# Grant CAN_QUERY on the KA's serving endpoint.
# The Permissions API requires the endpoint ID (not the name) in the URL path.
kas = ws.knowledge_assistants.list_knowledge_assistants()
ka_details = next(ka for ka in kas if ka.display_name == KA_NAME)
ka_endpoint_name = ka_details.endpoint_name
print(f"KA serving endpoint: {ka_endpoint_name}")

# Retrieve the serving endpoint ID from the endpoint details
ep_resp = _requests.get(
    f"{ws.config.host}/api/2.0/serving-endpoints/{ka_endpoint_name}",
    headers=_headers,
    timeout=30,
)
ep_resp.raise_for_status()
ka_endpoint_id = ep_resp.json()["id"]

resp = _requests.patch(
    f"{ws.config.host}/api/2.0/permissions/serving-endpoints/{ka_endpoint_id}",
    headers=_headers,
    json={
        "access_control_list": [
            {"group_name": "users", "permission_level": "CAN_QUERY"}
        ]
    },
    timeout=30,
)
resp.raise_for_status()
print(f"Granted CAN_QUERY on serving endpoint '{ka_endpoint_name}' (id={ka_endpoint_id}) to all users")

# Grant read-only access on shared catalog resources
# Users can browse the PDF and see the data but cannot modify anything
spark.sql(f"GRANT USE CATALOG ON CATALOG {SHARED_CATALOG} TO `account users`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {SHARED_CATALOG}.{SHARED_SCHEMA} TO `account users`")
spark.sql(f"GRANT READ VOLUME ON VOLUME {SHARED_CATALOG}.{SHARED_SCHEMA}.{SHARED_VOLUME} TO `account users`")
spark.sql(f"GRANT SELECT ON TABLE {SHARED_CATALOG}.{SHARED_SCHEMA}.{TABLE_NAME} TO `account users`")

print(f"Granted read-only access on {SHARED_CATALOG}.{SHARED_SCHEMA} to all users")

## 6. Wait for KA endpoint to come online

In [0]:
print(f"Waiting for KA endpoint to come online (this may take several minutes)...")
try:
    abm.ka_wait_until_endpoint_online(ka_tile_id, timeout_s=600)
    print(f"KA endpoint is ONLINE and ready for students.")
except TimeoutError as e:
    print(f"Warning: {e}")
    print("The KA may still be provisioning. Students can proceed once it's online.")

## Summary

In [0]:
print("=" * 60)
print("Workspace Setup Complete")
print("=" * 60)
print(f"  Shared catalog   : {SHARED_CATALOG}")
print(f"  Shared schema    : {SHARED_SCHEMA}")
print(f"  PDF location     : {pdf_path}")
print(f"  KA name          : {KA_NAME}")
print(f"  KA tile_id       : {ka_tile_id}")
print(f"  User permissions : READ-ONLY on shared resources")